### Structured Output
##### Structured output allows agents to return data in a specific, predictable format. Instead of parsing natural language responses, 
##### you get structured data in the form of JSON objects, Pydantic models, or dataclasses that your application can use directly.


### Pydentic - init_chat_model
#### Pydentic models provide the richest feature set with field validations, descriptions and nested structures.

In [ ]:
## import the os module and set the GROQ_API_KEY environment variable
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
## import the init_chat_model function from langchain.chat_models
from langchain.chat_models import init_chat_model
### initialize the chat model
### Groq Llama Chat model
model = init_chat_model(
    model="llama-3.3-70b-versatile",
    model_provider="groq"
)  # Groq model

model

In [ ]:
## import the BaseModel class from the pydantic module
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title: str=Field(description="The title of the movie")
    year: int=Field(description="The year the movie was released")
    director: str=Field(description="The director of the movie")
    rating: float=Field(description="The rating of the movie out of 10")

In [ ]:
## import the with_structured_output function from langchain.chat_models and use it to create a new model that outputs structured data in the form of a Movie object
model_with_structured_output = model.with_structured_output(Movie)

In [ ]:
## Default response from the model
defaultresponse = model.invoke("What is the movie Naya Daur about? Actor Didlip Kumar and actress Vyjayanthimala")
defaultresponse

In [ ]:
## Now Structured output from the model 
structuredresponse = model_with_structured_output.invoke("What is the movie Naya Daur about? Actor Didlip Kumar and actress Vyjayanthimala")
structuredresponse

In [ ]:
## import the BaseModel class from the pydantic module and define a new class called Movie that inherits from BaseModel. 
# The Movie class has four attributes: title, year, director, and rating. Each attribute is defined using the Field function from pydantic, 
# which allows you to specify additional metadata for each attribute, such as a description.
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title: str=Field(...,description="The title of the movie")
    year: int=Field(...,description="The year the movie was released")
    director: str=Field(...,description="The director of the movie")
    rating: float=Field(...,description="The rating of the movie out of 10")
## import the with_structured_output function from langchain.chat_models and use it to create a new model that outputs structured data in the form of a Movie object.
## The include_raw parameter is set to True, which means that the raw output from the model will also be included in the structured output.
model_with_structured_output = model.with_structured_output(Movie, include_raw=True)
response = model_with_structured_output.invoke("What is the movie Naya Daur about? Actor Didlip Kumar and actress Vyjayanthimala")
response

### Pydantic - Nested Structured Output

In [ ]:
from pydantic import BaseModel,Field
class Actor(BaseModel):
    name: str
    role: str
class MovieDetals(BaseModel):
    title: str
    year: int
    casts: list[Actor]
    budget: float | None =Field(None,description="The budget of the movie")

model_with_structure = model.with_structured_output(MovieDetals)
response = model_with_structure.invoke("What is the movie Naya Daur about? Actor Didlip Kumar and actress Vyjayanthimala")
response



### Pydantic - create_agent

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
from pydantic import BaseModel,Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

In [ ]:
result["structured_response"]

### TypeDict

In [ ]:
from typing_extensions import Annotated, TypedDict
class MovieDict(TypedDict):
    """A Movie with title, year, director, and rating."""
    title: Annotated[str,..., "The title of the movie"]
    year: Annotated[int,..., "The year the movie was released"]
    director: Annotated[str,..., "The director of the movie"]
    rating: Annotated[float,..., "The rating of the movie out of 10"]
modeld_with_TypeDict=model.with_structured_output(MovieDict)
response = modeld_with_TypeDict.invoke("Provide the details of movie Braveheart")
response

### TypeDict Nested

In [ ]:
from typing_extensions import Annotated, TypedDict
class Actor(TypedDict):
    name: str
    role: str
class MovieDetails(TypedDict):
    title: str
    year: int
    casts: list[Actor]
    budget: float | None =Field(None,description="The budget of the movie in inr")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("What is the movie Naya Daur about? Actor Didlip Kumar and actress Vyjayanthimala")
response

### DataClass

In [ ]:
from dataclasses import dataclass
from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]